FileFinder for AccessViz: This function finds a list of travel time matrix files based on a list of YKR ID values from a specified input data folder. The code works for different list lengths and different YKR ID values. If the YKR ID number does not exist in the input folder (and it’s subfolders), the tool will warn about this to the user but still continue running. The tool also informs the user about the execution process: tells the user what file is currently under process and how many files there are left (e.g. "Processing file travel_times_to_5797076.txt.. Progress: 3/25"). As output, FileFinder compiles a list of FilePaths for further processing. FileFinder can also print out a list of filepaths into a text file.

In [76]:
#import all the necessary modules

#import pathlib to manage file paths
import pathlib

#import regex
import re

#geopandas to read the YKR grid shapefile
import geopandas

#pandas to read the travel time matrix file
import pandas as pd

#imports for plotting
import matplotlib.pyplot as plt
import contextily
import folium


In [77]:
#filefinder function

def filefinder(id_list, directory):
    """ A function that finds a list of travel time matrix files based on
    a list of YKR ID values and the input data directory.
    The code works for different list lengths and different YKR ID values.
    The function will return an empty list if an empty list is passed on.
    The function will warn about duplicates and skip them.
    The function will also warn about missing files.
    A progress report will also be shown for each search.
    Input : A list of YKR IDs and the input directory
    Return: A list of travel time matrix file paths. """

    string_id_list = [str(num) for num in id_list]

    #get a list of filepaths from the input directory
    filepaths = list( directory.rglob("travel_times_to_*.txt"))

    #get a list of filenames from the filepaths using pathlib
    filenames = [x.name for x in filepaths]

    #make a dictionary of ykr ids and corresponding filepaths
    files_dictionary = {(re.search(r"\d+", key)).group():value for key, value in zip(filenames, filepaths)}

    #empty list to store the required file paths
    files_out =[]

    #empty list to store id values to check for duplicates
    ids_list = []

    #loop to search through the files
    for i in range (len(string_id_list)):

        #conditional statemnet to check for duplicates
        if (string_id_list[i]) in ids_list:
            print(f"The ID: {string_id_list[i]} is a duplicate, so it will be skipped... Progress {i+1}/{len(string_id_list)}")
            continue
        else:

            #try and except to catch errors
            try:
                print(f"Processing file: {files_dictionary[string_id_list[i]]} ... Progress {i+1}/{len(string_id_list)}")
                files_out.append(files_dictionary[string_id_list[i]])

                #processed ids will only be added if the file exits
                ids_list.append(string_id_list[i])

            except KeyError:
                print(f"The ID: {string_id_list[i]} did not have a matching file, Check the YKR Id again... Progress {i+1}/{len(string_id_list)}")
    
            
        
    return(files_out)
        



TableJoiner for AccessViz: This tool creates a spatial layer from the chosen Matrix text table (e.g. travel_times_to_5797076.txt) by joining the Matrix file with MetropAccess_YKR_grid Shapefile where from_id in Matrix file corresponds to YKR_ID in the Shapefile. The tool saves the result in the output-folder that user has defined. Output file format can be Shapefile or Geopackage. The files are named in a way that it is possible to identify the ID from the name (e.g. 5797076). The table joiing is applied to files that correspond to a list of selected YKR ID files (FileFinder handles finding the correct input files!).

In [78]:
#tablejoiner function
def tablejoiner(files, grid, Output_Directory, output_type):

    """ This function creates a spatial layer from the chosen Matrix text table 
    (e.g. travel_times_to_5797076.txt) by joining the Matrix file with MetropAccess_YKR_grid Shapefile 
    where from_id in Matrix file corresponds to YKR_ID in the Shapefile. 
    The tool saves the result in the output-folder that user has defined. 
    Output file format can be Shapefile or Geopackage. 
    The files are named in a way that it is possible to identify the ID from the name (e.g. 5797076). T
    he table joiing is applied to files that correspond to a list of selected YKR ID files."""
    
    #empty list to store the output file paths
    output_file_paths = []

    #loop through the files returned from filefinder and merge them each with the grid file
    for file_path in files:

        #read the .text travel file
        travel_data = pd.read_csv(file_path, sep=";")
        #rename to make sure both this file and grid have same column names to merge
        travel_data = travel_data.rename(columns={"from_id": "YKR_ID"})
        #merge
        travel_data_grid = grid.merge(travel_data, on='YKR_ID', how='left')

        #get the file name using regex
        out_file_name = (re.search(r"\d+", file_path.name)).group()

        #choose the extension based on users choice
        if output_type == "shp":
            out_file_name_full = f"{out_file_name}.shp"
        else:
            out_file_name_full = f"{out_file_name}.gpkg"
            
        #save the file
        travel_data_grid.to_file(Output_Directory/out_file_name_full)
        #append the file paths to return later
        output_file_paths.append(Output_Directory/out_file_name_full)

    return(output_file_paths)

    

Visualizer for AccessViz: This function can visualize the travel times of selected YKR_IDs based on different travel modes (it is possible to use the same tool for visualizing travel times by car, public transport, walking or biking depending on an input parameter!). It saves the maps into a specified folder for output images. The output maps can be either static or interactive - it is possible to select which kind of map output is generated when running the tool. The visualizer can be applied to files that correspond to a list of selected YKR ID files (FileFinder handles finding the correct input files!),

In [79]:

def _plot_static(travel_grid, column, name, travel_mode, Output_Directory):
    """Helper function to plot static maps."""

    #convert the crs top epsg 3857 for contextily base map
    travel_grid_3857 = travel_grid.to_crs(epsg=3857)

    #plot the travel time matrix grid
    ax = travel_grid_3857.plot(
        figsize=(12, 8),
    
        column= column,
        scheme="quantiles",
        cmap="Spectral",
        linewidth=0.5,
        alpha=0.8,
        legend=True,
        legend_kwds={"title": f"Travel time by {travel_mode} to {name}."}
        )

    #set the bounds for the ax object
    minx, miny, maxx, maxy = travel_grid_3857.total_bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    #add the base map
    contextily.add_basemap(ax, source=contextily.providers.OpenStreetMap.Mapnik)

    #get the file name for each unique YKR ID
    filename = f"{name}_{travel_mode}_static.png"

    #save the file
    plt.savefig(Output_Directory/filename)

    #close the plt object to save memory
    plt.close()


def _plot_interactive(travel_grid, column, name, travel_mode, Output_Directory):
    """Helper function to plot interactive maps."""
    
    #convert crs for folium
    travel_grid_4326 = travel_grid.to_crs(epsg=4326)

    #set the location and zoom for the interactive base map
    interactive_map = folium.Map(
        location=(60.2, 24.8),
        zoom_start=10,
        control_scale=True,
        tiles="cartodbpositron"
    )

    #add the travel time grid to the base map, make sure to set the right key and values columns for folium
    travel_grid_layer = folium.Choropleth(
        geo_data=travel_grid_4326,
        data=travel_grid_4326,
        columns= ("YKR_ID", column),
        bins=5,
        key_on="feature.properties.YKR_ID",
        fill_color="YlOrRd",
        line_weight=0,
        legend_name=f"Travel time by {travel_mode} to {name}.",
        highlight=True
    )
    travel_grid_layer.add_to(interactive_map)

    #get the file name and save the file
    filename = f"{name}_{travel_mode}_interactive.html"
    interactive_map.save(Output_Directory/filename)


def visualiser(out_files, columns, travel_mode, map_type, Output_Directory):
    """This function can visualize the travel times of selected YKR_IDs 
    based on different travel modes (it is possible to use the same tool 
    for visualizing travel times by car, public transport, walking or biking 
    depending on an input parameter!). It saves the maps into a specified folder 
    for output images. The output maps can be either static or interactive - 
    it is possible to select which kind of map output is generated when running 
    the tool. The visualizer can be applied to files that correspond to a list of s
    elected YKR ID files (FileFinder handles finding the correct input files!)"""

    for file_path in out_files:
        travel_grid = geopandas.read_file(file_path)
        name = (re.search(r"\d+", file_path.name)).group()
        column = columns[travel_mode]
        travel_grid = travel_grid[travel_grid[column] != -1]

        if map_type == 'static':
            _plot_static(travel_grid, column, name, travel_mode, Output_Directory)
        else:
            _plot_interactive(travel_grid, column, name, travel_mode, Output_Directory)



Comparison tool for AccessViz can compare travel times or travel distances between two different travel modes. For example, the tool can compare rush hour travel times by public transport and car based on columns pt_r_t and car_r_t, and rush hour travel distances based on columns pt_r_d and car_r_d. It is also possible to run the AccessViz tool without doing any comparisons. Thus IF the user has specified two travel modes (passed in as a list) for the AccessViz, the tool will calculate the time/distance difference of those travel modes into a new column. In the calculation, the first travel mode is always subtracted by the last one: travelmode1 - travelmode2 according to the order in which the travel modes were listed. The tool also ensures that distances are not compared to travel times and vice versa. The tool saves outputs as new files (Shapefile or Geopackage file format) with an informative name, for example: Accessibility_5797076_pt_vs_car.shp. It is also possible to compare only two travel modes between each other at the time. Accepted travel modes are the same ones that are found in the actual TravelTimeMatrix file (walking, biking, public transport and car). If the tool gets invalid parameters (for example, a travel mode that does not exists, or too many travel modes), it stops the program, and gives advice what are the acceptable values.



In [80]:
  

def comparison(out_files, modes, comp_type, output_dir ):
    """Comparison tool for AccessViz can compare travel times or travel distances 
    between two different travel modes. For example, the tool can compare rush hour 
    travel times by public transport and car based on columns pt_r_t and car_r_t, and 
    rush hour travel distances based on columns pt_r_d and car_r_d. It is also possible 
    to run the AccessViz tool without doing any comparisons. Thus IF the user has specified 
    two travel modes (passed in as a list) for the AccessViz, the tool will calculate the 
    time/distance difference of those travel modes into a new column. In the calculation, 
    the first travel mode is always subtracted by the last one: travelmode1 - travelmode2 
    according to the order in which the travel modes were listed. The tool also ensures that 
    distances are not compared to travel times and vice versa. The tool saves outputs as new files 
    (Shapefile or Geopackage file format) with an informative name, for example: 
    Accessibility_5797076_pt_vs_car.shp. It is also possible to compare only two travel
    modes between each other at the time. Accepted travel modes are the same ones that 
    are found in the actual TravelTimeMatrix file (walking, biking, public transport and car). 
    If the tool gets invalid parameters (for example, a travel mode that does not exists, or 
    too many travel modes), it stops the program, and gives advice what are the acceptable values."""

    #check if only 2 modes are provided
    if len(modes) != 2:
        raise ValueError(f"Exactly 2 travel modes must be provided, got {len(modes)}.")

    #valiate the type of comparison
    if comp_type not in ['time', 'distance']:
        raise ValueError(f"{comp_type} is invalid. Can only compare time or distance")

    #empty list to store the columns to be compared
    comp_columns = []

    #validate the travel modes using a lopp and add the columns to be compared on to the empty list
    for i in modes:
        if i not in time_columns.keys():
            raise ValueError(f"Mode {i} not found in travel data.")
        elif comp_type == 'time':
            comp_columns.append(time_columns[i])
        else:
            comp_columns.append(distance_columns[i])
    
    for file_path in out_files:
        travel_grid = geopandas.read_file(file_path)
        grid_name = (re.search(r"\d+", file_path.name)).group()
        column_name = f"{modes[0]}_{modes[1]}"

        travel_grid[column_name]=travel_grid[comp_columns[0]]-travel_grid[comp_columns[1]]
        
        #mask to get rows with -1 values
        mask = (travel_grid[comp_columns[0]] == -1) | (travel_grid[comp_columns[1]] == -1)

        #replace the new column values with -1
        travel_grid.loc[mask, column_name] = -1
            
        comp_file_name = f"Accessibility_{grid_name}_{modes[0]}_vs_{modes[1]}_{comp_type}.shp"
        travel_grid.to_file(output_dir/comp_file_name)

        _plot_static(travel_grid, column_name, grid_name, f"{column_name}_difference", Output_Image_Directory)
 

AccessViz Tool

In [81]:
#set notebook path
NOTEBOOK_PATH = pathlib.Path().resolve()

#set the input data directory
Data_Directory = NOTEBOOK_PATH / "data"

#list of YKR IDs to be used for the analysis
id_list = [5785640, 5785641, 5785642]

#call filefinder
files = filefinder(id_list, Data_Directory)

#set the output directory for the merged files
Output_Directory = NOTEBOOK_PATH / "output"

#set the type of output files: :gpkg" or "shp"
output_type = "shp"

#read the shapefile 
grid = geopandas.read_file(Data_Directory/"MetropAccess_YKR_grid/MetropAccess_YKR_grid")

#call tablejoiner function
out_files = tablejoiner(files, grid, Output_Directory, output_type)

#set the output directory for saving the plots
Output_Image_Directory = NOTEBOOK_PATH / "images"

#select travel mode for visualisation: (car, pt or walk)
travel_mode = 'car'

#set the type of map: static or interactive
map_type = 'interactive'

#identify the corresponding column names. This current one is taken from Helsinki Region Travel Time Matrix 2013
#Travel Time Matrix files from other years can have more columns, edit the dictionary based on the respective file
columns = {"car": "car_m_t", "pt": "pt_m_t", "walk": "walk_t"}

#call the visualiser function
visualiser(out_files, columns, travel_mode, map_type, Output_Image_Directory)

#set the output directory for saving comparisons
Comp_Output_Directory = NOTEBOOK_PATH / "output"

#select travel modes that needs to be compared: (car, pt or walk)
modes = ['car', 'pt']

#set the comparison type: time or distance
comp_type = 'time'

#set the time and distance columns as per the travel time matrix
time_columns = {"car": "car_m_t", "pt": "pt_m_t", "walk": "walk_t"}
distance_columns = {"car": "car_m_d", "pt": "pt_m_d", "walk": "walk_d"}

#call the comparison function       
comparison(out_files, modes, comp_type, Comp_Output_Directory )
    




Processing file: C:\Users\Rahul M\Documents\Projects\AccessViz\data\HelsinkiRegion_TravelTimeMatrix2013\5785xxx\travel_times_to_ 5785640.txt ... Progress 1/3
Processing file: C:\Users\Rahul M\Documents\Projects\AccessViz\data\HelsinkiRegion_TravelTimeMatrix2013\5785xxx\travel_times_to_ 5785641.txt ... Progress 2/3
Processing file: C:\Users\Rahul M\Documents\Projects\AccessViz\data\HelsinkiRegion_TravelTimeMatrix2013\5785xxx\travel_times_to_ 5785642.txt ... Progress 3/3
